In [1]:
]activate ../../../../

  Activating project at `~/UCLOneDrive/SpatialStructureInMicrobialCommunities/SSMCCode`


In [2]:
using Revise
includet("./base.jl")

┌ Warning: Circular dependency detected.
│ Precompilation will be skipped for dependencies in this cycle:
│  ┌ Symbolics → SymbolicsForwardDiffExt
│  └─ Symbolics → SymbolicsPreallocationToolsExt
│ Precompilation will also be skipped for the following, which depend on the above cycle:
│   SSMC
│   MLSolver
└ @ Pkg.API.Precompilation ~/.julia/juliaup/julia-1.10.10+0.x64.linux.gnu/share/julia/stdlib/v1.10/Pkg/src/precompilation.jl:583
Precompiling packages...
  13000.4 ms  ✓ SSMCMain
  1 dependency successfully precompiled in 17 seconds. 544 already precompiled. 4 skipped due to circular dependency.


In [3]:
using CairoMakie
using GLMakie
CairoMakie.activate!()

In [4]:
f = jldopen("./sys1.jld2")

JLDFile /home/honza/UCLOneDrive/SpatialStructureInMicrobialCommunities/SSMCCode/cluster_env/runs/si_totbiom/run1system/sys1.jld2 (read-only)
 ├─🔢 gen_ps
 ├─🔢 ode_s
 └─🔢 ode_fs

In [21]:
gen_ps = f["gen_ps"]
ode_s = f["ode_s"]
ode_fs = f["ode_fs"]

N, M = get_Ns(gen_ps)
iri = findfirst(!iszero, gen_ps.K)

16

In [ ]:
T = 1e8
L = 5
sN = 1000
sp_epsilon = 1e-3
tol = 1e-9
dx = L / sN

pde_u0 = expand_u0_to_size((sN,), ode_fs)
pde_u0 = perturb_u0_uniform(N, M, pde_u0, sp_epsilon)
clamp!(pde_u0, 0., Inf);

In [26]:
function change_p(ps, p; DI=1.)
    N, M = get_Ns(ps)
    iri = findfirst(!iszero, gen_ps.K)

    new_ps = copy(ps);
    new_ps.Ds[N+1:N+M] .= p * DI
    new_ps.Ds[N+iri] = DI

    new_ps
end

change_p (generic function with 1 method)

In [ ]:
ps = 10 .^ range(0, -2, 10)
sols = Vector{Any}(undef, length(ps));

for i in 1:length(ps)
    p = ps[i]

    local_ps = change_p(gen_ps, p)

    pde_ps = BSMMiCRMParams(
        local_ps.mmicrm_params,
        local_ps.Ds,
        CartesianSpace{1,Tuple{Periodic}}(SA[dx]),
        solver_threads
    )
    pde_p = make_smmicrm_problem(pde_ps, pde_u0, T; jac_type=:sparse);

    pde_s_t, pde_s_u, scb = make_stepped_saver_callback(pde_u0, 10)
    pde_s = solve(pde_p, TRBDF2();
        dense=false,
        abstol=tol,
        reltol=tol,
        callback=CallbackSet(make_timer_callback(5), PositiveDomain(copy(pde_u0); save=false), scb),
    );
    smmicrmmaxresid(pde_s)
end

In [ ]:
p = 0.01
local_ps = change_p(gen_ps, p)

pde_ps = BSMMiCRMParams(
    gen_ps.mmicrm_params,
    gen_ps.Ds,
    CartesianSpace{1,Tuple{Periodic}}(SA[dx]),
    nthreads()
)
pde_p = make_smmicrm_problem(pde_ps, pde_u0, T; jac_type=:sparse);

pde_s_t, pde_s_u, scb = make_stepped_saver_callback(pde_u0, 10)
pde_s = solve(pde_p, TRBDF2();
    dense=false,
    abstol=tol,
    reltol=tol,
    callback=CallbackSet(make_timer_callback(5), PositiveDomain(copy(pde_u0); save=false), scb),
);
smmicrmmaxresid(pde_s)

135.27718680631426